# FIFA 21 Data Cleaning — V1 + V2

**Goal:** Load the two FIFA 21 raw CSV versions separately, inspect and clean each one, standardize their different column names, validate the results, and then combine them into one analysis-ready dataset.

This notebook follows the class workflow: **Inspect → Clean → Validate → Visualise → Export**.

> Important: V1 and V2 describe the same FIFA 21 player universe at different dataset versions. We therefore **concatenate rows** after standardizing the schema, rather than doing a side-by-side SQL-style merge that would create `_x` and `_y` copies of most columns. A `Dataset_Version` column is kept so the source of every row is clear.

## Setup
The Kaggle dataset contains two raw CSV files with different schemas:

- **V1:** `fifa21_raw_data.csv`
- **V2:** `fifa21 raw data v2.csv`

The loader below first tries KaggleHub and then falls back to a local file with the same name if it is already beside the notebook.

In [ ]:
!pip -q install -U kagglehub

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_colwidth', 120)

print('Libraries imported successfully.')

### Load V1 and V2 separately
Using separate variables prevents one raw version from accidentally overwriting the other.

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

DATASET_HANDLE = 'yagunnersya/fifa-21-messy-raw-dataset-for-cleaning-exploring'
V1_FILE = 'fifa21_raw_data.csv'
V2_FILE = 'fifa21 raw data v2.csv'


def load_fifa_file(file_name):
    """Load one CSV from Kaggle; if that fails, try the local working directory."""
    try:
        return kagglehub.dataset_load(
            KaggleDatasetAdapter.PANDAS,
            DATASET_HANDLE,
            file_name,
            pandas_kwargs={'low_memory': False}
        )
    except Exception as kaggle_error:
        if os.path.exists(file_name):
            print(f'KaggleHub unavailable; loading local file: {file_name}')
            return pd.read_csv(file_name, low_memory=False)
        raise RuntimeError(
            f'Could not load {file_name} from Kaggle and no local copy was found. '
            f'Original error: {kaggle_error}'
        )


df_raw_v1 = load_fifa_file(V1_FILE)
df_raw_v2 = load_fifa_file(V2_FILE)

print('V1 shape:', df_raw_v1.shape)
print('V2 shape:', df_raw_v2.shape)

# Stage 1 — Inspect both raw datasets
We inspect shape, sample rows, data types, missing values, duplicates, and column-name differences before changing anything.

In [ ]:
print('V1 first 3 rows:')
display(df_raw_v1.head(3))

print('V2 first 3 rows:')
display(df_raw_v2.head(3))

In [ ]:
print('--- V1 INFO ---')
df_raw_v1.info()

print()
print('--- V2 INFO ---')
df_raw_v2.info()

In [ ]:
print('V1 duplicate rows:', df_raw_v1.duplicated().sum())
print('V2 duplicate rows:', df_raw_v2.duplicated().sum())

print()
print('V1 columns with missing values:')
display(df_raw_v1.isnull().sum().loc[lambda s: s > 0].sort_values(ascending=False).to_frame('Missing'))

print()
print('V2 columns with missing values:')
display(df_raw_v2.isnull().sum().loc[lambda s: s > 0].sort_values(ascending=False).to_frame('Missing'))

In [ ]:
# Compare column names after only removing the down-arrow symbol and outer spaces.
def normalized_column_set(df):
    return set(df.columns.astype(str).str.replace('↓', '', regex=False).str.strip())

v1_cols = normalized_column_set(df_raw_v1)
v2_cols = normalized_column_set(df_raw_v2)

print('Only in V1:', sorted(v1_cols - v2_cols))
print('Only in V2:', sorted(v2_cols - v1_cols))

### Main schema mismatch
V1 uses a combined `Team & Contract` field and older names such as `foot` and `BP`. V2 uses separate `Club` and `Contract` columns plus `Preferred Foot` and `Best Position`.

We will convert V1 to the V2 naming scheme **before** combining the datasets.

In [ ]:
COLUMN_MAPPING_V1_TO_V2 = {
    'foot': 'Preferred Foot',
    'BP': 'Best Position'
}

display(pd.DataFrame({
    'V1 column': ['Team & Contract', 'foot', 'BP', 'Growth'],
    'V2 equivalent': ['Club + Contract (split)', 'Preferred Foot', 'Best Position', 'Not present in V2'],
    'Action': ['Split into 2 columns', 'Rename', 'Rename', 'Drop before combination']
}))

### Data Quality Report
This focuses on the problems listed in the assignment: line breaks, text amounts, mixed units, star symbols, `Hits`, dates, and duplicates.

In [ ]:
def count_contains(series, pattern):
    s = series.astype('string')
    return int(s.str.contains(pattern, regex=True, na=False).sum())


def make_quality_report(df, version):
    rows = []

    club_col = 'Team & Contract' if 'Team & Contract' in df.columns else 'Club'
    if club_col in df.columns:
        rows.append([club_col, 'Embedded line breaks / extra whitespace', count_contains(df[club_col], r'[\r\n]')])

    for col in ['Value', 'Wage', 'Release Clause']:
        if col in df.columns:
            rows.append([col, 'Currency sign and/or K/M suffix stored as text', count_contains(df[col], r'[€KM]')])

    if 'Height' in df.columns:
        rows.append(['Height', 'Mixed text units / non-numeric representation', int(pd.to_numeric(df['Height'], errors='coerce').isna().sum())])

    if 'Weight' in df.columns:
        rows.append(['Weight', 'Mixed text units / non-numeric representation', int(pd.to_numeric(df['Weight'], errors='coerce').isna().sum())])

    for col in ['W/F', 'SM', 'IR']:
        if col in df.columns:
            rows.append([col, 'Star symbol stored with rating', count_contains(df[col], r'★')])

    if 'Hits' in df.columns:
        affected = int(df['Hits'].isna().sum()) + count_contains(df['Hits'], r'[Kk]')
        rows.append(['Hits', 'Missing values and/or K suffix', affected])

    if 'Joined' in df.columns:
        rows.append(['Joined', 'Date stored as text', int(df['Joined'].notna().sum())])

    rows.append(['Entire row', 'Exact duplicates', int(df.duplicated().sum())])

    report = pd.DataFrame(rows, columns=['Column(s)', 'What is wrong', 'Rows affected'])
    report.insert(0, 'Dataset', version)
    return report

quality_v1 = make_quality_report(df_raw_v1, 'V1')
quality_v2 = make_quality_report(df_raw_v2, 'V2')
quality_report = pd.concat([quality_v1, quality_v2], ignore_index=True)
display(quality_report)

# Stage 2 — Clean V1 and V2 separately
The functions below are reused for both datasets so the same cleaning rules are applied consistently.

## 2.0 Standardize the V1 schema
This step splits `Team & Contract`, renames the older columns, and removes `Growth` because V2 does not contain an equivalent field.

In [ ]:
def standardize_schema(df, version):
    df = df.copy()
    df.columns = df.columns.astype(str).str.replace('↓', '', regex=False).str.strip()

    if version == 'V1':
        # Split the old combined Team & Contract field at its final line break.
        if 'Team & Contract' in df.columns:
            tc = (df['Team & Contract']
                  .astype('string')
                  .str.replace('\r', '', regex=False)
                  .str.replace(r'\n+', '\n', regex=True)
                  .str.strip())

            split_tc = tc.str.rsplit('\n', n=1, expand=True)

            # Normal case: final line is the contract; everything before it is the club.
            if split_tc.shape[1] >= 2:
                club = split_tc.iloc[:, 0].astype('string').str.strip()
                contract = split_tc.iloc[:, 1].astype('string').str.strip()
            else:
                club = tc.copy()
                contract = pd.Series(pd.NA, index=df.index, dtype='string')

            # Fallback for any row where no line-break split was obtained.
            contract_pattern = r'(\d{4}\s*~\s*\d{4}|Free|[A-Za-z]{3}\s+\d{1,2},\s*(?:\d{4}\s*)?On Loan)$'
            missing_contract = contract.isna() | contract.eq('')
            if missing_contract.any():
                extracted = tc[missing_contract].str.extract(contract_pattern, expand=False)
                contract.loc[missing_contract] = extracted
                club.loc[missing_contract] = (
                    tc[missing_contract]
                    .str.replace(contract_pattern, '', regex=True)
                    .str.strip()
                )

            df['Club'] = club
            df['Contract'] = contract
            df.drop(columns=['Team & Contract'], inplace=True)

        df.rename(columns=COLUMN_MAPPING_V1_TO_V2, inplace=True)

        # Growth is a V1-only field, so remove it to make V1 compatible with V2.
        if 'Growth' in df.columns:
            df.drop(columns=['Growth'], inplace=True)

    return df

v1_schema = standardize_schema(df_raw_v1, 'V1')
v2_schema = standardize_schema(df_raw_v2, 'V2')

print('Columns only in standardized V1:', sorted(set(v1_schema.columns) - set(v2_schema.columns)))
print('Columns only in standardized V2:', sorted(set(v2_schema.columns) - set(v1_schema.columns)))

## 2.1–2.9 Cleaning helpers
- Clean text and whitespace
- Convert money to numeric values
- Convert height to cm
- Convert weight to kg
- Remove star symbols
- Convert/fill `Hits`
- Parse `Joined` and create `Years_at_Club`
- Flag wage outliers with IQR
- Remove exact duplicate rows

In [ ]:
def parse_compact_number(value):
    """Convert values such as €67.5M, €220K, 1.6K, 500 into floats."""
    if pd.isna(value):
        return np.nan

    s = str(value).strip().upper().replace('€', '').replace(',', '')
    if s in {'', 'NAN', 'NONE', '<NA>', '-'}:
        return np.nan

    multiplier = 1
    if s.endswith('M'):
        multiplier = 1_000_000
        s = s[:-1]
    elif s.endswith('K'):
        multiplier = 1_000
        s = s[:-1]

    try:
        return float(s) * multiplier
    except ValueError:
        return np.nan


def parse_height_cm(value):
    """Convert 175cm or 5'11" to centimeters."""
    if pd.isna(value):
        return np.nan

    s = str(value).strip().lower().replace(' ', '')

    if s.endswith('cm'):
        return round(float(s[:-2]), 1)

    feet_inches = re.match(r'^(\d+)[\'′](\d+)(?:["″])?$', s)
    if feet_inches:
        feet = float(feet_inches.group(1))
        inches = float(feet_inches.group(2))
        return round(feet * 30.48 + inches * 2.54, 1)

    # Fallback if a numeric cm value already exists.
    try:
        return round(float(s), 1)
    except ValueError:
        return np.nan


def parse_weight_kg(value):
    """Convert 72kg or 159lbs to kilograms."""
    if pd.isna(value):
        return np.nan

    s = str(value).strip().lower().replace(' ', '')

    if s.endswith('kg'):
        return round(float(s[:-2]), 1)
    if s.endswith('lbs'):
        return round(float(s[:-3]) * 0.453592, 1)
    if s.endswith('lb'):
        return round(float(s[:-2]) * 0.453592, 1)

    try:
        return round(float(s), 1)
    except ValueError:
        return np.nan


def clean_one_version(raw_df, version):
    df = standardize_schema(raw_df, version)
    original_rows = len(df)

    # 2.1 Clean all text columns: line breaks, repeated spaces, leading/trailing spaces.
    text_cols = df.select_dtypes(include=['object', 'string']).columns
    for col in text_cols:
        df[col] = (
            df[col].astype('string')
            .str.replace(r'[\r\n]+', ' ', regex=True)
            .str.replace(r'\s+', ' ', regex=True)
            .str.strip()
        )

    # 2.2 Amounts
    for col in ['Value', 'Wage', 'Release Clause']:
        if col in df.columns:
            df[col] = df[col].apply(parse_compact_number)

    # 2.3 Height -> cm
    if 'Height' in df.columns:
        df['Height'] = df['Height'].apply(parse_height_cm)

    # 2.4 Weight -> kg
    if 'Weight' in df.columns:
        df['Weight'] = df['Weight'].apply(parse_weight_kg)

    # 2.5 Star ratings -> plain numbers
    for col in ['W/F', 'SM', 'IR']:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col].astype('string').str.extract(r'(\d+)', expand=False),
                errors='coerce'
            )

    # 2.6 Hits -> number; median is robust to very large popular-player values.
    if 'Hits' in df.columns:
        df['Hits'] = df['Hits'].apply(parse_compact_number)
        hits_median = df['Hits'].median()
        df['Hits'] = df['Hits'].fillna(hits_median)
    else:
        hits_median = np.nan

    # 2.7 Joined -> datetime and Years_at_Club as of 1 Jan 2021.
    if 'Joined' in df.columns:
        df['Joined'] = pd.to_datetime(df['Joined'], errors='coerce')
        reference_date = pd.Timestamp('2021-01-01')
        df['Years_at_Club'] = ((reference_date - df['Joined']).dt.days / 365.25).round(1)

    # 2.8 Wage outliers using IQR. Keep the rows; only flag them.
    if 'Wage' in df.columns:
        q1, q3 = df['Wage'].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower_fence = q1 - 1.5 * iqr
        upper_fence = q3 + 1.5 * iqr
        df['Wage_Outlier'] = (df['Wage'] < lower_fence) | (df['Wage'] > upper_fence)
    else:
        lower_fence = upper_fence = np.nan

    # 2.9 Remove only exact duplicate rows.
    duplicates_removed = int(df.duplicated().sum())
    df = df.drop_duplicates().reset_index(drop=True)

    # Keep version origin before the final combination.
    df.insert(0, 'Dataset_Version', version)

    cleaning_stats = {
        'version': version,
        'original_rows': original_rows,
        'clean_rows': len(df),
        'duplicates_removed': duplicates_removed,
        'hits_median_used': hits_median,
        'wage_lower_fence': lower_fence,
        'wage_upper_fence': upper_fence
    }

    return df, cleaning_stats

## Clean V1

In [ ]:
df_v1_clean, stats_v1 = clean_one_version(df_raw_v1, 'V1')
print(stats_v1)
display(df_v1_clean.head())

## Clean V2

In [ ]:
df_v2_clean, stats_v2 = clean_one_version(df_raw_v2, 'V2')
print(stats_v2)
display(df_v2_clean.head())

### Quick checks after cleaning
The lower wage fence can be negative even though a wage cannot be negative. That simply means the IQR rule does not identify any meaningful low-end outliers; practically, the relevant wage outliers are at the high end.

In [ ]:
print('V1 zero Value rows:', int((df_v1_clean['Value'] == 0).sum()))
print('V1 zero Wage rows:', int((df_v1_clean['Wage'] == 0).sum()))
print('V2 zero Value rows:', int((df_v2_clean['Value'] == 0).sum()))
print('V2 zero Wage rows:', int((df_v2_clean['Wage'] == 0).sum()))

print()
print('V1 wage fences:', stats_v1['wage_lower_fence'], 'to', stats_v1['wage_upper_fence'])
print('V2 wage fences:', stats_v2['wage_lower_fence'], 'to', stats_v2['wage_upper_fence'])

In [ ]:
print('Top 10 earners in V1:')
display(df_v1_clean.nlargest(10, 'Wage')[['Name', 'Club', 'Wage', 'Wage_Outlier']])

print('Top 10 earners in V2:')
display(df_v2_clean.nlargest(10, 'Wage')[['Name', 'Club', 'Wage', 'Wage_Outlier']])

In [ ]:
print('Players with more than 10 years at their club — V1:')
display(df_v1_clean.loc[df_v1_clean['Years_at_Club'] > 10,
                        ['Name', 'Club', 'Joined', 'Years_at_Club']]
        .sort_values('Years_at_Club', ascending=False)
        .head(30))

print('Players with more than 10 years at their club — V2:')
display(df_v2_clean.loc[df_v2_clean['Years_at_Club'] > 10,
                        ['Name', 'Club', 'Joined', 'Years_at_Club']]
        .sort_values('Years_at_Club', ascending=False)
        .head(30))

# Stage 3 — Validate each cleaned dataset
These assertions intentionally stop the notebook if a required cleaning rule failed.

In [ ]:
def validate_cleaned(df, raw_df, stats):
    treated = ['Value', 'Wage', 'Height', 'Weight', 'Hits']

    # Required treated fields must be complete and numeric.
    assert df[treated].isnull().sum().sum() == 0, 'Null value remains in a treated column.'
    for col in treated:
        assert pd.api.types.is_numeric_dtype(df[col]), f'{col} is not numeric.'

    # Reasonable physical ranges from the assignment.
    assert df['Height'].between(150, 220).all(), 'Height outside 150–220 cm.'
    assert df['Weight'].between(50, 120).all(), 'Weight outside 50–120 kg.'

    # Star columns should be 1–5 after conversion.
    for col in ['W/F', 'SM', 'IR']:
        assert df[col].dropna().between(1, 5).all(), f'{col} has a value outside 1–5.'

    # Row-count change is allowed only for exact duplicates removed in step 2.9.
    expected_rows = len(raw_df) - stats['duplicates_removed']
    assert len(df) == expected_rows, 'Unexpected row-count change.'

    assert pd.api.types.is_datetime64_any_dtype(df['Joined']), 'Joined is not datetime.'
    assert df['Dataset_Version'].nunique() == 1

    print(f"{stats['version']} validation PASSED — {len(df):,} cleaned rows.")


validate_cleaned(df_v1_clean, df_raw_v1, stats_v1)
validate_cleaned(df_v2_clean, df_raw_v2, stats_v2)

# Combine V1 and V2
Now that both datasets are independently clean, align their columns and concatenate them.

If an unexpected column exists in only one version, this cell prints it and adds an empty column to the other version rather than silently losing information.

In [ ]:
def align_columns(df1, df2):
    df1 = df1.copy()
    df2 = df2.copy()

    # Prefer V2 order because V2 is the newer schema; append any V1-only extras at the end.
    final_columns = list(df2.columns) + [c for c in df1.columns if c not in df2.columns]

    for col in final_columns:
        if col not in df1.columns:
            df1[col] = pd.NA
        if col not in df2.columns:
            df2[col] = pd.NA

    return df1[final_columns], df2[final_columns]

v1_aligned, v2_aligned = align_columns(df_v1_clean, df_v2_clean)

print('Columns only in cleaned V1 before alignment:', sorted(set(df_v1_clean.columns) - set(df_v2_clean.columns)))
print('Columns only in cleaned V2 before alignment:', sorted(set(df_v2_clean.columns) - set(df_v1_clean.columns)))

# Row-wise combination after schema alignment.
df = pd.concat([v1_aligned, v2_aligned], ignore_index=True)

print('Combined shape:', df.shape)
display(df.head())

### Validate the combined table
We **do not drop cross-version duplicates by `ID`** because V1 and V2 are separate dataset versions and may contain updated values for the same player. `Dataset_Version` preserves that history.

In [ ]:
assert len(df) == len(df_v1_clean) + len(df_v2_clean)
assert set(df['Dataset_Version'].dropna().unique()) == {'V1', 'V2'}
assert df[['Value', 'Wage', 'Height', 'Weight', 'Hits']].isnull().sum().sum() == 0
assert df['Height'].between(150, 220).all()
assert df['Weight'].between(50, 120).all()

print('Combined validation PASSED.')
print(df['Dataset_Version'].value_counts())

# Stage 4 — Visualise
The three plots below follow the assignment requirements. Each uses the cleaned combined table while retaining version labels where useful.

## Plot 1 — Raw height representation vs cleaned height
Simply deleting letters/symbols from mixed height strings is wrong: for example, `5'11"` becomes `511`, which is not centimeters. Proper unit conversion places realistic player heights between about 150 and 220 cm.

In [ ]:
raw_heights = pd.concat([
    df_raw_v1['Height'].astype('string'),
    df_raw_v2['Height'].astype('string')
], ignore_index=True)

raw_digits_only = pd.to_numeric(
    raw_heights.str.replace(r'[^0-9.]', '', regex=True),
    errors='coerce'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(raw_digits_only.dropna(), bins=35)
axes[0].axvline(150, linestyle='--')
axes[0].axvline(220, linestyle='--')
axes[0].set_title('Raw Height After Naively Deleting Letters/Symbols')
axes[0].set_xlabel('Raw numeric-looking value')
axes[0].set_ylabel('Players')

axes[1].hist(df['Height'].dropna(), bins=35)
axes[1].axvline(150, linestyle='--')
axes[1].axvline(220, linestyle='--')
axes[1].set_title('Clean Height in Centimeters')
axes[1].set_xlabel('Height (cm)')
axes[1].set_ylabel('Players')

plt.tight_layout()
plt.show()

## Plot 2 — Wage distribution
The normal-scale histogram is strongly right-skewed because a small group of elite players earns much more than most players. A log x-axis spreads out the lower and middle wage ranges and makes the overall distribution easier to inspect.

In [ ]:
combined_upper_fence = df['Wage'].quantile(0.75) + 1.5 * (
    df['Wage'].quantile(0.75) - df['Wage'].quantile(0.25)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Wage'], bins=50)
axes[0].axvline(combined_upper_fence, linestyle='--')
axes[0].set_title('Wage Distribution — Normal Axis')
axes[0].set_xlabel('Wage (€)')
axes[0].set_ylabel('Players')

positive_wage = df.loc[df['Wage'] > 0, 'Wage']
axes[1].hist(positive_wage, bins=50)
axes[1].axvline(combined_upper_fence, linestyle='--')
axes[1].set_xscale('log')
axes[1].set_title('Wage Distribution — Log X Axis')
axes[1].set_xlabel('Wage (€), log scale')
axes[1].set_ylabel('Players')

plt.tight_layout()
plt.show()

## Plot 3 — Player value vs wage
Both variables span wide ranges, so log scales make their relationship easier to see. Wage-outlier players are plotted separately so unusually high earners are easy to identify.

In [ ]:
plot_df = df[(df['Value'] > 0) & (df['Wage'] > 0)].copy()

normal = plot_df[~plot_df['Wage_Outlier']]
outlier = plot_df[plot_df['Wage_Outlier']]

plt.figure(figsize=(9, 6))
plt.scatter(normal['Value'], normal['Wage'], alpha=0.35, label='Normal wage')
plt.scatter(outlier['Value'], outlier['Wage'], alpha=0.75, label='Wage outlier')
plt.xscale('log')
plt.yscale('log')
plt.title('Player Value vs Wage')
plt.xlabel('Value (€) — log scale')
plt.ylabel('Wage (€) — log scale')
plt.legend()
plt.tight_layout()
plt.show()

# Export cleaned files
The two version-specific files make it clear that V1 and V2 were cleaned independently. The final assignment file is the combined dataset.

In [ ]:
df_v1_clean.to_csv('fifa21_v1_cleaned.csv', index=False)
df_v2_clean.to_csv('fifa21_v2_cleaned.csv', index=False)
df.to_csv('fifa21_cleaned_data.csv', index=False)

print('Saved:')
print(' - fifa21_v1_cleaned.csv')
print(' - fifa21_v2_cleaned.csv')
print(' - fifa21_cleaned_data.csv  <-- final combined file')

In [ ]:
# Re-open the final CSV once to confirm it survived export correctly.
check_export = pd.read_csv('fifa21_cleaned_data.csv', low_memory=False)
print('Exported shape:', check_export.shape)
check_export.info()

# Conclusion
Both FIFA 21 raw dataset versions were loaded and inspected separately before any cleaning was performed. V1 required schema repair because `Team & Contract` combined two fields and it used older names such as `foot` and `BP`; these were standardized to the V2 schema. Text whitespace, financial amounts, height, weight, star ratings, `Hits`, and the `Joined` date were cleaned using reusable functions. Missing `Hits` values were filled with the median because it is less sensitive to extreme popularity values than the mean, while wage outliers were flagged rather than deleted. Exact duplicate rows were removed only within each version. The two cleaned datasets were then column-aligned and concatenated with a `Dataset_Version` field so their origin remains traceable. Before training a model, I would also check whether keeping both versions of the same player could create data leakage or overweight players who appear in both versions.